In [67]:
from transformers import GPT2LMHeadModel, GPT2TokenizerFast
import torch
import json
from datasets import Dataset
import numpy as np
from transformers import Trainer, TrainingArguments

In [68]:
model_name = 'openai-community/gpt2'

In [69]:
model = GPT2LMHeadModel.from_pretrained(model_name)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: openai-community/gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [70]:
model

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [71]:
tokenizer = GPT2TokenizerFast.from_pretrained(model_name)

In [72]:
len(tokenizer)

50257

In [73]:
promt = 'Hi'

In [74]:
tokenizer.encode(promt)

[17250]

In [75]:
tokenizer(promt)

{'input_ids': [17250], 'attention_mask': [1]}

In [76]:
tokenizer(promt, return_tensors='pt')

{'input_ids': tensor([[17250]]), 'attention_mask': tensor([[1]])}

In [77]:
model(input_ids=torch.tensor([[17250]]))

CausalLMOutputWithCrossAttentions(loss=None, logits=tensor([[[-34.2417, -34.3302, -37.3032,  ..., -43.0447, -42.7168, -35.2204]]],
       grad_fn=<UnsafeViewBackward0>), past_key_values=DynamicCache(layers=[DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer]), hidden_states=None, attentions=None, cross_attentions=None)

In [78]:
enc = tokenizer(promt, return_tensors='pt')
enc

{'input_ids': tensor([[17250]]), 'attention_mask': tensor([[1]])}

In [79]:
output = model(**enc)
output

CausalLMOutputWithCrossAttentions(loss=None, logits=tensor([[[-34.2417, -34.3302, -37.3032,  ..., -43.0447, -42.7168, -35.2204]]],
       grad_fn=<UnsafeViewBackward0>), past_key_values=DynamicCache(layers=[DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer]), hidden_states=None, attentions=None, cross_attentions=None)

In [80]:
output.logits

tensor([[[-34.2417, -34.3302, -37.3032,  ..., -43.0447, -42.7168, -35.2204]]],
       grad_fn=<UnsafeViewBackward0>)

In [81]:
output.logits.shape

torch.Size([1, 1, 50257])

In [82]:
output.logits.squeeze()

tensor([-34.2417, -34.3302, -37.3032,  ..., -43.0447, -42.7168, -35.2204],
       grad_fn=<SqueezeBackward0>)

In [83]:
output.logits.squeeze().argmax()

tensor(13)

In [84]:
output.logits[0][-1].argmax()

tensor(13)

In [85]:
tokenizer.decode(torch.tensor(13))

'.'

In [86]:
out = model.generate(enc['input_ids'], max_length=100)

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


In [87]:
out

tensor([[17250,    13,   314,  1101,  7926,    11,   475,   314,  1101,   407,
          1654,   611,   345,   821,  3910,   286,   428,    13,   314,  1101,
           407,  1654,   611,   345,   821,  3910,   286,   428,    13,   198,
           198,    40,  1101,  7926,    11,   475,   314,  1101,   407,  1654,
           611,   345,   821,  3910,   286,   428,    13,   314,  1101,   407,
          1654,   611,   345,   821,  3910,   286,   428,    13,   198,   198,
            40,  1101,  7926,    11,   475,   314,  1101,   407,  1654,   611,
           345,   821,  3910,   286,   428,    13,   314,  1101,   407,  1654,
           611,   345,   821,  3910,   286,   428,    13,   198,   198,    40,
          1101,  7926,    11,   475,   314,  1101,   407,  1654,   611,   345]])

In [88]:
tokenizer.decode(out[0])

"Hi. I'm sorry, but I'm not sure if you're aware of this. I'm not sure if you're aware of this.\n\nI'm sorry, but I'm not sure if you're aware of this. I'm not sure if you're aware of this.\n\nI'm sorry, but I'm not sure if you're aware of this. I'm not sure if you're aware of this.\n\nI'm sorry, but I'm not sure if you"

In [89]:
promt = 'Who is Elon Mask'

In [90]:
enc = tokenizer(promt, return_tensors='pt')
enc

{'input_ids': tensor([[ 8241,   318, 32451, 18007]]), 'attention_mask': tensor([[1, 1, 1, 1]])}

In [91]:
out = model.generate(enc['input_ids'], max_length=50)

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


In [92]:
tokenizer.decode(out[0])

'Who is Elon Masked?"\n\n"I\'m Elon Masked," he said. "I\'m a guy who\'s been in the business for a long time. I\'m a guy who\'s been in the business for a long time. I'

In [93]:
data = json.load(open('instruction-data.json'))

In [94]:
data[0]

{'instruction': 'Evaluate the following phrase by transforming it into the spelling given.',
 'input': 'freind --> friend',
 'output': 'The spelling of the given phrase "freind" is incorrect, the correct spelling is "friend".'}

In [95]:
len(data)

1100

In [96]:
USER_TOKEN = '<user>'
ASSISTANT_TOKEN = '<assistant>'
MAX_LENGTH = 512

In [97]:
def format_input(example):
  instruction = example['instruction']
  input = example.get('input', '').strip()
  output = example['output']
  if input:
    user_text = f"{instruction}\n\n{input}"
  else:
    user_text = instruction
  text = f"{USER_TOKEN}{user_text}{ASSISTANT_TOKEN}{output}"
  return {'text':text}

In [98]:
exp_data = format_input(data[0])

In [99]:
exp_data['text']

'<user>Evaluate the following phrase by transforming it into the spelling given.\n\nfreind --> friend<assistant>The spelling of the given phrase "freind" is incorrect, the correct spelling is "friend".'

In [100]:
tokenizer.special_tokens_map

{'bos_token': '<|endoftext|>',
 'eos_token': '<|endoftext|>',
 'unk_token': '<|endoftext|>'}

In [101]:
dataset = Dataset.from_list([format_input(example) for example in data])

In [102]:
dataset

Dataset({
    features: ['text'],
    num_rows: 1100
})

In [103]:
len(tokenizer.vocab)

50257

In [104]:
tokenizer.eos_token

'<|endoftext|>'

In [105]:
tokenizer.add_special_tokens({
    'pad_token': tokenizer.eos_token,
    'additional_special_tokens':[USER_TOKEN, ASSISTANT_TOKEN]
})

3

In [106]:
len(tokenizer.vocab)

50259

In [107]:
tokenizer.convert_tokens_to_ids(USER_TOKEN)

50257

In [108]:
example = dataset[0]['text']
example

'<user>Evaluate the following phrase by transforming it into the spelling given.\n\nfreind --> friend<assistant>The spelling of the given phrase "freind" is incorrect, the correct spelling is "friend".'

In [109]:
def prepare_input(text):
  enc = tokenizer(example,
                truncation = True,
                max_length = 100,
                padding = 'max_length'
                )
  input_ids = enc['input_ids']
  labels = [-100]*len(input_ids)
  assistant_id = tokenizer.convert_tokens_to_ids(ASSISTANT_TOKEN)
  pad_id = tokenizer.convert_tokens_to_ids(tokenizer.pad_token)
  start = input_ids.index(assistant_id)
  stop = input_ids.index(pad_id) + 1
  for i in range(start, stop):
    labels[i] = input_ids[i]
  return {
      'input_ids': input_ids,
      'attention_mask': enc['attention_mask'],
      'labels': labels
  }

In [110]:
tokenized_ds = dataset.map(prepare_input, remove_columns='text')
tokenized_ds

Map:   0%|          | 0/1100 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 1100
})

In [111]:
enc = tokenizer(example,
                truncation = True,
                max_length = 100,
                padding = 'max_length'
                )
enc

{'input_ids': [50257, 36, 2100, 4985, 262, 1708, 9546, 416, 25449, 340, 656, 262, 24993, 1813, 13, 198, 198, 19503, 521, 14610, 1545, 50258, 464, 24993, 286, 262, 1813, 9546, 366, 19503, 521, 1, 318, 11491, 11, 262, 3376, 24993, 318, 366, 6726, 1911, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]}

In [112]:
input_ids = enc['input_ids']

In [113]:
labels = [-100]*len(input_ids)

In [114]:
assistant_id = tokenizer.convert_tokens_to_ids(ASSISTANT_TOKEN)
pad_id = tokenizer.convert_tokens_to_ids(tokenizer.pad_token)

In [115]:
np.array(input_ids)

array([50257,    36,  2100,  4985,   262,  1708,  9546,   416, 25449,
         340,   656,   262, 24993,  1813,    13,   198,   198, 19503,
         521, 14610,  1545, 50258,   464, 24993,   286,   262,  1813,
        9546,   366, 19503,   521,     1,   318, 11491,    11,   262,
        3376, 24993,   318,   366,  6726,  1911, 50256, 50256, 50256,
       50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
       50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
       50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
       50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
       50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
       50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
       50256])

In [116]:
np.array(labels)

array([-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
       -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
       -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
       -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
       -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
       -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
       -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
       -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
       -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
       -100])

In [117]:
assistant_id

50258

In [118]:
pad_id

50256

In [119]:
start = input_ids.index(assistant_id) + 1
stop = input_ids.index(pad_id) + 1

In [120]:
for i in range(start, stop):
  labels[i] = input_ids[i]

In [121]:
training_args = TrainingArguments(
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs = 4,
    optim = 'adamw_torch',
    report_to = 'none',
    warmup_steps=200,
    lr_scheduler_type='cosine'
)

In [122]:
train = Trainer(
    model=model,
    args = training_args,
    train_dataset=tokenized_ds,

)

In [123]:
model.resize_token_embeddings

<bound method PreTrainedModel.resize_token_embeddings of GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)>

In [124]:
model.transformer.wte.weight

Parameter containing:
tensor([[-0.1101, -0.0393,  0.0331,  ..., -0.1364,  0.0151,  0.0453],
        [ 0.0403, -0.0486,  0.0462,  ...,  0.0861,  0.0025,  0.0432],
        [-0.1275,  0.0479,  0.1841,  ...,  0.0899, -0.1297, -0.0879],
        ...,
        [-0.0445, -0.0548,  0.0123,  ...,  0.1044,  0.0978, -0.0695],
        [ 0.1860,  0.0167,  0.0461,  ..., -0.0963,  0.0785, -0.0225],
        [ 0.0514, -0.0277,  0.0499,  ...,  0.0070,  0.1552,  0.1207]],
       device='cuda:0', requires_grad=True)

In [125]:
model.resize_token_embeddings(len(tokenizer))

Embedding(50259, 768)

In [126]:
model.transformer.wte.weight

Parameter containing:
tensor([[-0.1101, -0.0393,  0.0331,  ..., -0.1364,  0.0151,  0.0453],
        [ 0.0403, -0.0486,  0.0462,  ...,  0.0861,  0.0025,  0.0432],
        [-0.1275,  0.0479,  0.1841,  ...,  0.0899, -0.1297, -0.0879],
        ...,
        [ 0.0514, -0.0277,  0.0499,  ...,  0.0070,  0.1552,  0.1207],
        [-0.0025, -0.0585,  0.1174,  ...,  0.0236,  0.0039,  0.0344],
        [-0.0026, -0.0585,  0.1174,  ...,  0.0236,  0.0039,  0.0344]],
       device='cuda:0', requires_grad=True)

In [127]:
model

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50259, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [129]:
train.train()

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
500,0.302227


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=552, training_loss=0.2737623412268358, metrics={'train_runtime': 205.6308, 'train_samples_per_second': 21.398, 'train_steps_per_second': 2.684, 'total_flos': 224547840000000.0, 'train_loss': 0.2737623412268358, 'epoch': 4.0})

In [131]:
train.save_model('./gpt2_instruction_fine_tune')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [133]:
from google.colab import drive

In [134]:
drive.mount('drive')

Mounted at drive


In [137]:
!cp -r '/content/gpt2_instruction_fine_tune' '/content/drive/MyDrive/Colab/fine-tune-models'